# [Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html#full-implementation)

## Import torch packages

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

## Load datasets

In [5]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

## Create the model

In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## Hyperparameters
Hyperparameters are adjustable parameters that let you control the model optimization process. Different hyperparameter values can impact model training and convergence rates (read more about hyperparameter tuning)

We define the following hyperparameters for training:

- Number of Epochs - the number of times to iterate over the dataset

- Batch Size - the number of data samples propagated through the network before the parameters are updated

- Learning Rate - how much to update models parameters at each batch/epoch. Smaller values yield slow learning speed, while large values may result in unpredictable behavior during training.



In [7]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

## Optimization loop

### Loss Function

In [8]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

### Optimizer

In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

### Training loop

In [10]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

### Evaluation loop

In [11]:
def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

### Run the loop

In [12]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.297392  [   64/60000]
loss: 2.281287  [ 6464/60000]
loss: 2.263750  [12864/60000]
loss: 2.265135  [19264/60000]
loss: 2.228968  [25664/60000]
loss: 2.207106  [32064/60000]
loss: 2.212608  [38464/60000]
loss: 2.171544  [44864/60000]
loss: 2.166852  [51264/60000]
loss: 2.142719  [57664/60000]
Test Error: 
 Accuracy: 53.4%, Avg loss: 2.130664 

Epoch 2
-------------------------------
loss: 2.139863  [   64/60000]
loss: 2.130202  [ 6464/60000]
loss: 2.065003  [12864/60000]
loss: 2.081737  [19264/60000]
loss: 2.021892  [25664/60000]
loss: 1.959175  [32064/60000]
loss: 1.979830  [38464/60000]
loss: 1.896735  [44864/60000]
loss: 1.894785  [51264/60000]
loss: 1.811002  [57664/60000]
Test Error: 
 Accuracy: 61.8%, Avg loss: 1.820321 

Epoch 3
-------------------------------
loss: 1.855386  [   64/60000]
loss: 1.830316  [ 6464/60000]
loss: 1.700856  [12864/60000]
loss: 1.739651  [19264/60000]
loss: 1.634403  [25664/60000]
loss: 1.589437  [32064/600